# TP Jour 1 — Stockage Distribué HDFS, Acquisition Multi-Datasets & Couche Bronze

Bienvenue dans ce premier volet pratique de notre architecture **Data Lakehouse** !

### Objectifs de la séance
1. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) et vérifier sa configuration (`dfs.blocksize = 16 Mo`, `dfs.replication = 2`).
2. **Récupérer deux datasets Kaggle complémentaires** dans `/workspace/data_raw/<source>/` : **Olist Brazilian E-Commerce** (commandes B2C) et **Olist Marketing Funnel** (acquisition B2B des vendeurs, joignable sur `seller_id`).
3. **Ingérer les fichiers bruts multi-sources sur HDFS** et observer physiquement le découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : simuler la perte d'un DataNode et observer l'auto-réparation (ré-allocation automatique des blocs sous-répliqués en ~50 secondes).
5. **Construire la couche Bronze** (`/lakehouse/bronze/`) avec PySpark 4.2 + Delta Lake 4.4 (métadonnées `_ingested_at`, `_source_dataset`, `_source_file`) et inspecter notre premier DAG sur **Spark Web UI** (`http://localhost:4040`).

---
### Interfaces utiles pendant le TP
| Service | URL | Identifiant |
|---|---|---|
| **HDFS NameNode UI** | http://localhost:9870 | — |
| **JupyterLab** | http://localhost:8888 | Token : `bigdata` |
| **Spark Web UI** | http://localhost:4040 | *(Actif dès que `SparkSession` démarre)* |
| **Dashboard Streamlit** | http://localhost:8501 | — |

## 1. Acquisition Multi-Datasets : Olist E-Commerce (B2C) + Olist Marketing Funnel (B2B)

Nous combinons **deux datasets publics Kaggle** partageant la clé métier `seller_id` :
1. **`olistbr/brazilian-ecommerce`** : ~100 000 commandes réelles au Brésil sur 9 tables relationnelles (~120 Mo au total, dont `olist_geolocation_dataset.csv` d'environ **61 Mo**, idéal pour observer le découpage en blocs HDFS de **16 Mo**).
2. **`olistbr/marketing-funnel-olist`** : entonnoir d'acquisition B2B des marchands Olist (2 tables : *Marketing Qualified Leads* `mql_id` / canal `origin`, et *Closed Deals* reliant `mql_id` à `seller_id` et `business_segment`).

Chaque source est rangée dans son propre sous-dossier `data_raw/<nom-dataset>/` afin d'éviter toute collision de noms.

In [ ]:
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASETS = {
    "brazilian-ecommerce": "olistbr/brazilian-ecommerce",
    "marketing-funnel-olist": "olistbr/marketing-funnel-olist",
}

# Rétrocompatibilité : si les CSV Olist étaient déjà à la racine de data_raw/, on les range dans brazilian-ecommerce/
ecom_dir = DATA_RAW / "brazilian-ecommerce"
root_ecom_csvs = [f for f in DATA_RAW.glob("*.csv") if f.name.startswith(("olist_", "product_category_"))]
if len(root_ecom_csvs) >= 8 and not ecom_dir.exists():
    ecom_dir.mkdir(parents=True, exist_ok=True)
    for f in root_ecom_csvs:
        shutil.move(str(f), str(ecom_dir / f.name))

for subdir, slug in KAGGLE_DATASETS.items():
    target_dir = DATA_RAW / subdir
    existing = sorted(target_dir.glob("*.csv")) if target_dir.exists() else []
    if existing:
        print(f"[{subdir}] déjà présent dans {target_dir} ({len(existing)} fichiers CSV).")
    else:
        import kagglehub
        print(f"Téléchargement de {slug} depuis Kaggle Hub -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for csv_file in sorted(cache_dir.rglob("*.csv")):
            shutil.copy2(csv_file, target_dir / csv_file.name)

existing_csvs = sorted(DATA_RAW.rglob("*.csv"))
print(f"\nInventaire multi-datasets local ({len(existing_csvs)} fichiers CSV) :")
for f in existing_csvs:
    rel = f.relative_to(DATA_RAW).as_posix()
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {rel:<62} {size_mb:>6.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

> **Secours Hors-Ligne (uniquement si le Wi-Fi bloque l'accès externe à Kaggle)** :
> Décommentez et exécutez la cellule ci-dessous **uniquement** si le téléchargement `kagglehub` échoue faute de connexion Internet. Elle génère localement un jeu synthétique respectant exactement le schéma Olist (dont un fichier `olist_geolocation_dataset.csv` > 40 Mo pour tester le découpage multi-blocs).

In [ ]:
# === CELLULE DE SECOURS HORS-LIGNE (Garder commentée si Kaggle a fonctionné) ===
# import numpy as np, pandas as pd
# if len(list(DATA_RAW.glob("*.csv"))) < 8:
#     rng = np.random.default_rng(42)
#     n_orders, n_cust, n_prod, n_sell = 25_000, 20_000, 1_500, 400
#     states = ["SP", "RJ", "MG", "RS", "PR", "SC", "BA", "PE", "CE", "PA", "GO", "DF"]
#     state_p = [0.42, 0.13, 0.12, 0.06, 0.05, 0.04, 0.04, 0.03, 0.03, 0.02, 0.03, 0.03]
#     cats_pt = ["cama_mesa_banho", "beleza_saude", "esporte_lazer", "informatica_acessorios",
#                "moveis_decoracao", "utilidades_domesticas", "relogios_presentes", "telefonia"]
#     cats_en = ["bed_bath_table", "health_beauty", "sports_leisure", "computers_accessories",
#                "furniture_decor", "housewares", "watches_gifts", "telephony"]
#     pd.DataFrame({"product_category_name": cats_pt, "product_category_name_english": cats_en}).to_csv(
#         DATA_RAW / "product_category_name_translation.csv", index=False)
#     prod_ids = [f"prod_{i:05d}" for i in range(n_prod)]
#     pd.DataFrame({"product_id": prod_ids, "product_category_name": rng.choice(cats_pt, n_prod),
#                   "product_weight_g": rng.integers(100, 12000, n_prod)}).to_csv(
#         DATA_RAW / "olist_products_dataset.csv", index=False)
#     cust_ids = [f"cust_{i:06d}" for i in range(n_orders)]
#     uniq_ids = rng.choice([f"uniq_{i:06d}" for i in range(n_cust)], n_orders)
#     pd.DataFrame({"customer_id": cust_ids, "customer_unique_id": uniq_ids,
#                   "customer_zip_code_prefix": rng.integers(1000, 99999, n_orders),
#                   "customer_city": "sao paulo", "customer_state": rng.choice(states, n_orders, p=state_p)}).to_csv(
#         DATA_RAW / "olist_customers_dataset.csv", index=False)
#     sell_ids = [f"sell_{i:04d}" for i in range(n_sell)]
#     pd.DataFrame({"seller_id": sell_ids, "seller_zip_code_prefix": rng.integers(1000, 99999, n_sell),
#                   "seller_city": "sao paulo", "seller_state": rng.choice(states, n_sell, p=state_p)}).to_csv(
#         DATA_RAW / "olist_sellers_dataset.csv", index=False)
#     order_ids = [f"ord_{i:06d}" for i in range(n_orders)]
#     base_ts = pd.Timestamp("2017-01-01") + pd.to_timedelta(rng.integers(0, 600 * 24, n_orders), unit="h")
#     deliv_days = np.round(rng.lognormal(mean=2.3, sigma=0.55, size=n_orders), 1)
#     est_days = rng.integers(14, 32, n_orders)
#     pd.DataFrame({
#         "order_id": order_ids, "customer_id": cust_ids,
#         "order_status": rng.choice(["delivered", "shipped", "canceled"], n_orders, p=[0.96, 0.025, 0.015]),
#         "order_purchase_timestamp": base_ts, "order_approved_at": base_ts + pd.Timedelta(hours=2),
#         "order_delivered_carrier_date": base_ts + pd.Timedelta(days=2),
#         "order_delivered_customer_date": base_ts + pd.to_timedelta(deliv_days, unit="D"),
#         "order_estimated_delivery_date": base_ts + pd.to_timedelta(est_days, unit="D"),
#     }).to_csv(DATA_RAW / "olist_orders_dataset.csv", index=False)
#     prices = np.round(rng.lognormal(4.3, 0.75, n_orders), 2)
#     freights = np.round(rng.uniform(8.0, 45.0, n_orders), 2)
#     pd.DataFrame({"order_id": order_ids, "order_item_id": 1, "product_id": rng.choice(prod_ids, n_orders),
#                   "seller_id": rng.choice(sell_ids, n_orders), "shipping_limit_date": base_ts + pd.Timedelta(days=5),
#                   "price": prices, "freight_value": freights}).to_csv(
#         DATA_RAW / "olist_order_items_dataset.csv", index=False)
#     pd.DataFrame({"order_id": order_ids, "payment_sequential": 1,
#                   "payment_type": rng.choice(["credit_card", "boleto", "voucher", "debit_card"], n_orders, p=[0.74, 0.19, 0.05, 0.02]),
#                   "payment_installments": rng.integers(1, 10, n_orders),
#                   "payment_value": np.round(prices + freights, 2)}).to_csv(
#         DATA_RAW / "olist_order_payments_dataset.csv", index=False)
#     scores = np.clip(5 - (deliv_days > est_days).astype(int) * rng.integers(1, 4, n_orders), 1, 5)
#     pd.DataFrame({"review_id": [f"rev_{i:06d}" for i in range(n_orders)], "order_id": order_ids,
#                   "review_score": scores, "review_comment_title": "", "review_comment_message": "",
#                   "review_creation_date": base_ts + pd.to_timedelta(deliv_days + 1, unit="D"),
#                   "review_answer_timestamp": base_ts + pd.to_timedelta(deliv_days + 2, unit="D")}).to_csv(
#         DATA_RAW / "olist_order_reviews_dataset.csv", index=False)
#     n_geo = 750_000  # ~42 Mo -> 3 blocs HDFS de 16 Mo
#     pd.DataFrame({"geolocation_zip_code_prefix": rng.integers(1000, 99999, n_geo),
#                   "geolocation_lat": rng.uniform(-33.0, 5.0, n_geo),
#                   "geolocation_lng": rng.uniform(-73.0, -34.0, n_geo),
#                   "geolocation_city": "sao paulo environment",
#                   "geolocation_state": rng.choice(states, n_geo)}).to_csv(
#         DATA_RAW / "olist_geolocation_dataset.csv", index=False)
#     print("Dataset synthétique Olist généré avec succès !")

## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf` monté dans le conteneur, aucune configuration *boilerplate* n'est requise : Spark pointe nativement vers `hdfs://namenode:9000` et active **Delta Lake**.

Dès l'exécution de la cellule ci-dessous, ouvrez **http://localhost:4040** dans un nouvel onglet pour voir l'interface **Spark Web UI**.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J1_HDFS_Ingestion_Bronze")
    .getOrCreate()
)

print("Spark version       :", spark.version)
print("Default FileSystem  :", spark.conf.get("spark.hadoop.fs.defaultFS"))
print("Warehouse directory :", spark.conf.get("spark.sql.warehouse.dir"))

## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

Nous utilisons l'API `org.apache.hadoop.fs.FileSystem` de la JVM embarquée par PySpark pour téléverser les fichiers CSV de `/workspace/data_raw` vers `hdfs://namenode:9000/lakehouse/bronze/raw_csv/` et inspecter directement la localisation physique des blocs sur nos **3 DataNodes** (`datanode1`, `datanode2`, `datanode3`).

In [ ]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw_csv")
fs.mkdirs(raw_hdfs_dir)

for csv_path in sorted(DATA_RAW.rglob("*.csv")):
    rel_posix = csv_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{csv_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw_csv/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

In [ ]:
# Inspection physique récursive du découpage en blocs HDFS (16 Mo) et des réplicas (facteur 2 sur 3 DataNodes)
print("=== Répartition physique des blocs HDFS pour chaque fichier ===\n")
remote_iter = fs.listFiles(raw_hdfs_dir, True)
all_statuses = []
while remote_iter.hasNext():
    all_statuses.append(remote_iter.next())

for status in sorted(all_statuses, key=lambda s: s.getLen(), reverse=True):
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw_csv/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    replication = status.getReplication()
    block_size_mb = status.getBlockSize() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | blocksize={block_size_mb:.0f} Mo | repl={replication})")
    for idx, blk in enumerate(blocks):
        hosts = list(blk.getHosts())
        blk_mb = blk.getLength() / (1024 * 1024)
        print(f"   Bloc #{idx} ({blk_mb:5.2f} Mo, offset={blk.getOffset():>10}) -> DataNodes : {hosts}")
    print()

### Questions d'observation (à vérifier aussi sur http://localhost:9870)
1. Sur `http://localhost:9870`, ouvrez **Utilities > Browse the file system** puis naviguez dans `/lakehouse/bronze/raw_csv/`.
2. Pourquoi `olist_geolocation_dataset.csv` est-il découpé en plusieurs blocs alors que `olist_sellers_dataset.csv` n'occupe qu'un seul bloc (sans pour autant consommer 16 Mo complets sur disque) ?
3. Vérifiez que chaque bloc possède **exactement 2 réplicas** placés sur 2 DataNodes distincts parmi `{datanode1, datanode2, datanode3}`.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans un cluster Hadoop par défaut, le NameNode attend **10 minutes et 30 secondes** avant de déclarer un DataNode mort. Dans notre fichier `config/hdfs-site.xml`, nous avons abaissé ce délai à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole de manipulation (depuis un terminal sur votre machine hôte) :
1. Vérifiez d'abord en CLI HDFS l'état sain des blocs :
   ```bash
   docker exec namenode hdfs fsck /lakehouse/bronze/raw_csv/brazilian-ecommerce/olist_geolocation_dataset.csv -files -blocks -locations
   ```
2. **Tuez brutalement le `datanode2`** :
   ```bash
   docker stop datanode2
   # (ou via mise : mise run crash-dn2)
   ```
3. Rafraîchissez l'onglet **Datanodes** sur `http://localhost:9870` toutes les 10 secondes :
   - Le compteur `Last contact` de `datanode2` grimpe (10s, 25s, 45s...).
   - Autour de **50 secondes**, `datanode2` passe à l'état **Dead** !
   - Les blocs qu'il hébergeait deviennent brièvement `Under-Replicated Blocks`, puis le NameNode ordonne immédiatement au DataNode survivant possédant une copie d'en envoyer une seconde vers le 3e DataNode actif !
4. Exécutez la cellule ci-dessous **pendant que `datanode2` est éteint** pour prouver que la lecture PySpark est 100 % transparente et sans interruption de service :

In [ ]:
# Vérification de la résilience : lecture intégrale de olist_geolocation_dataset.csv même avec un DataNode éteint !
GEO_HDFS_PATH = "/lakehouse/bronze/raw_csv/brazilian-ecommerce/olist_geolocation_dataset.csv"
df_geo_check = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(GEO_HDFS_PATH)
)

print(f"Nombre total de lignes lues depuis HDFS : {df_geo_check.count():,}")
print(f"Nombre de partitions Spark créées (1 par bloc HDFS de 16 Mo !) : {df_geo_check.rdd.getNumPartitions()}")

# Inspection de la nouvelle localisation des blocs après auto-réparation
geo_status = fs.getFileStatus(PathHDFS(GEO_HDFS_PATH))
for idx, blk in enumerate(fs.getFileBlockLocations(geo_status, 0, geo_status.getLen())):
    print(f"Bloc #{idx} -> réplicas actifs : {list(blk.getHosts())}")

> **Remarque fondamentale :** Observez `df_geo_check.rdd.getNumPartitions()` ! Par défaut, Spark crée **une partition de calcul par bloc HDFS de 16 Mo** (*Data Locality*). C'est ainsi que la taille de bloc HDFS pilote directement le parallélisme initial de Spark !
>
> N'oubliez pas de rallumer `datanode2` une fois l'observation terminée :
> ```bash
> docker start datanode2
> ```

---
## 5. Construction de la Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

Dans une architecture **Médaillon (Bronze / Silver / Gold)** :
- La couche **Bronze** conserve les données **fidèles à la source** (sans filtrage destructif), mais les convertit dans un format colonnaire transactionnel (**Delta Lake**) enrichi de métadonnées techniques de traçabilité (`_ingested_at`, `_source_file`).

In [ ]:
BRONZE_TABLES = {
    # Dataset #1 : olistbr/brazilian-ecommerce (B2C)
    "orders": ("brazilian-ecommerce", "brazilian-ecommerce/olist_orders_dataset.csv"),
    "order_items": ("brazilian-ecommerce", "brazilian-ecommerce/olist_order_items_dataset.csv"),
    "order_payments": ("brazilian-ecommerce", "brazilian-ecommerce/olist_order_payments_dataset.csv"),
    "order_reviews": ("brazilian-ecommerce", "brazilian-ecommerce/olist_order_reviews_dataset.csv"),
    "customers": ("brazilian-ecommerce", "brazilian-ecommerce/olist_customers_dataset.csv"),
    "products": ("brazilian-ecommerce", "brazilian-ecommerce/olist_products_dataset.csv"),
    "sellers": ("brazilian-ecommerce", "brazilian-ecommerce/olist_sellers_dataset.csv"),
    "category_translation": ("brazilian-ecommerce", "brazilian-ecommerce/product_category_name_translation.csv"),
    "geolocation": ("brazilian-ecommerce", "brazilian-ecommerce/olist_geolocation_dataset.csv"),
    # Dataset #2 : olistbr/marketing-funnel-olist (B2B acquisition vendeurs -> jointure sur seller_id)
    "marketing_leads": ("marketing-funnel-olist", "marketing-funnel-olist/olist_marketing_qualified_leads_dataset.csv"),
    "closed_deals": ("marketing-funnel-olist", "marketing-funnel-olist/olist_closed_deals_dataset.csv"),
}

for table_name, (source_dataset, csv_relpath) in BRONZE_TABLES.items():
    hdfs_csv_path = f"/lakehouse/bronze/raw_csv/{csv_relpath}"
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    df_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(hdfs_csv_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(source_dataset))
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    row_count = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] /lakehouse/bronze/{table_name:<22} (source={source_dataset:<22}) -> {row_count:>9,} lignes")

In [ ]:
# Vérification rapide d'un aperçu de la table Bronze `orders`
spark.read.format("delta").load("/lakehouse/bronze/orders").select(
    "order_id", "customer_id", "order_status", "order_purchase_timestamp", "_ingested_at"
).show(5, truncate=False)

### Bilan du Jour 1
- Vos 9 tables Olist sont stockées sur **HDFS** et converties en **Delta Lake** dans `/lakehouse/bronze/`.
- Comparez sur `http://localhost:9870` la taille disque des fichiers CSV dans `/lakehouse/bronze/raw_csv/` et des fichiers Parquet compressés Snappy dans `/lakehouse/bronze/geolocation/` : quel facteur de compression observez-vous ?
- Rendez-vous dans le notebook **`J2_Spark_Delta_Silver.ipynb`** pour le nettoyage, l'optimisation des jointures distribuées et les transactions ACID !